In [1]:
# ═══════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ═══════════════════════════════════════════════════════════════════
import os, sys, time, warnings, glob
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    BASE_DIR       = '/kaggle/input/birdclef-2026'
    MODEL_DIR      = '/kaggle/input/datasets/brandonkhuu/birdclef-2026-baseline-effb0-onnx'  
    TEST_DIR       = '/kaggle/input/competitions/birdclef-2026/test_soundscapes'
    SAMPLE_SUB     = '/kaggle/input/competitions/birdclef-2026/sample_submission.csv'
    
else:
    BASE_DIR       = 'data/raw'
    MODEL_DIR      = 'experiments'
    TEST_DIR       = os.path.join(BASE_DIR, 'test_soundscapes')
    SAMPLE_SUB     = os.path.join(BASE_DIR, 'sample_submission.csv')
    

print(f'Environment: {"Kaggle" if ON_KAGGLE else "Local"} ')
print(f'Test dir:    {TEST_DIR}')
print(f'Model dir:   {MODEL_DIR}')

Environment: Kaggle 
Test dir:    /kaggle/input/competitions/birdclef-2026/test_soundscapes
Model dir:   /kaggle/input/datasets/brandonkhuu/birdclef-2026-baseline-effb0-onnx


In [2]:
# ═══════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES (if needed)
# ═══════════════════════════════════════════════════════════════════
try:
    import onnxruntime as ort
except ImportError:
    import subprocess
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-deps', '-q',
        '/kaggle/input/datasets/brandonkhuu/onnx-runtime-whl/onnxruntime_wheel/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl'
    ])
    import onnxruntime as ort

print(f'onnxruntime: {ort.__version__}')

import librosa
print(f'librosa:     {librosa.__version__}')
print(f'numpy:       {np.__version__}')

onnxruntime: 1.24.4
librosa:     0.11.0
numpy:       2.0.2


In [3]:
# ═══════════════════════════════════════════════════════════════════
# SPECTROGRAM CONFIGURATION — matches training pipeline exactly
# ═══════════════════════════════════════════════════════════════════
SAMPLE_RATE      = 32000
N_MELS           = 128
FMAX             = 16000
HOP_LENGTH       = 512
N_FFT            = 2048
WINDOW_SECONDS   = 5.0
WINDOW_SAMPLES   = int(SAMPLE_RATE * WINDOW_SECONDS)  # 160000
NUM_CLASSES      = 234
SPEC_TIME_FRAMES = 313

# Load species list from sample submission (defines column order)
sample_sub = pd.read_csv(SAMPLE_SUB)
SPECIES_LIST = [c for c in sample_sub.columns if c != 'row_id']
assert len(SPECIES_LIST) == NUM_CLASSES, \
    f'Expected {NUM_CLASSES} species, got {len(SPECIES_LIST)}'
print(f'Species: {NUM_CLASSES}')
print(f'Sample submission rows: {len(sample_sub)}')

Species: 234
Sample submission rows: 3


In [4]:
# ═══════════════════════════════════════════════════════════════════
# LOAD ONNX MODELS
# ═══════════════════════════════════════════════════════════════════
def find_onnx_models(model_dir, n_folds=5):
    """Find ONNX model files, preferring regular ONNX (quantized is broken for this model)."""
    model_dir = model_dir if isinstance(model_dir, str) else str(model_dir)
    paths = []
    
    # Strategy 1: fold directories
    for fold_id in range(n_folds):
        candidates = [
            os.path.join(model_dir, f'baseline_effb0_fold{fold_id}', 'best_model.onnx'),
            os.path.join(model_dir, f'baseline_effb0_fold{fold_id}', 'best_model_quantized.onnx'),
            os.path.join(model_dir, f'fold{fold_id}.onnx'),
            os.path.join(model_dir, f'fold{fold_id}_quantized.onnx'),
            os.path.join(model_dir, f'best_model_fold{fold_id}.onnx'),
            os.path.join(model_dir, f'best_model_fold{fold_id}_quantized.onnx'),
            os.path.join(model_dir, f'best_model_{fold_id}.onnx'),
        ]
        for c in candidates:
            if os.path.exists(c):
                paths.append(c)
                break
    
    # Strategy 2: glob
    if not paths:
        all_onnx = sorted(glob.glob(os.path.join(model_dir, '**', '*.onnx'), recursive=True))
        regular = [f for f in all_onnx if 'quantized' not in f]
        quantized = [f for f in all_onnx if 'quantized' in f]
        paths = (regular if regular else quantized)[:n_folds]
    
    return paths

model_paths = find_onnx_models(MODEL_DIR)
print(f'Found {len(model_paths)} ONNX models:')

sessions = []
for p in model_paths:
    sess = ort.InferenceSession(p, providers=['CPUExecutionProvider'])
    sessions.append(sess)
    size_mb = os.path.getsize(p) / 1024 / 1024
    print(f'  {os.path.basename(p)}: {size_mb:.1f} MB')

N_MODELS = len(sessions)
print(f'\nEnsemble size: {N_MODELS} models')

Found 5 ONNX models:
  best_model_0.onnx: 17.1 MB
  best_model_1.onnx: 17.1 MB
  best_model_2.onnx: 17.1 MB
  best_model_3.onnx: 17.1 MB
  best_model_4.onnx: 17.1 MB

Ensemble size: 5 models


In [5]:
# ═══════════════════════════════════════════════════════════════════
# INFERENCE FUNCTIONS — with 3× Test-Time Augmentation (offset TTA)
#
# For each 5-second window, we predict on three time-shifted versions:
#   -2.5s offset, 0 offset, +2.5s offset
# and average the sigmoid probabilities. This catches bird calls that
# would be clipped at the boundaries of a single window.
#
# Cost: ~3× original inference time for the model calls, but audio
# loading and per-window spec compute are only slightly more expensive
# because spec compute (not audio I/O) is small relative to inference.
# ═══════════════════════════════════════════════════════════════════

TTA_OFFSETS_SAMPLES = [
    -int(0.5 * WINDOW_SAMPLES),  # -2.5 s
    0,                           #  center (original)
    +int(0.5 * WINDOW_SAMPLES),  # +2.5 s
]


def compute_melspec(waveform):
    """Compute log-mel spectrogram matching training pipeline."""
    S = librosa.feature.melspectrogram(
        y=waveform, sr=SAMPLE_RATE,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmax=FMAX,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    # Pad or trim to exact target size
    if S_db.shape[1] >= SPEC_TIME_FRAMES:
        S_db = S_db[:, :SPEC_TIME_FRAMES]
    else:
        S_db = np.pad(
            S_db, ((0, 0), (0, SPEC_TIME_FRAMES - S_db.shape[1])),
            mode='constant', constant_values=S_db.min(),
        )
    return S_db


def extract_shifted_segment(waveform, center_start, offset_samples):
    """
    Extract a 5-second segment from `waveform`, centered on the original
    window but shifted by `offset_samples`. Zero-pads the ends if the
    shifted window extends outside the waveform.
    """
    total_samples = len(waveform)
    start = center_start + offset_samples
    end = start + WINDOW_SAMPLES

    # Compute how much we need to pad on each side if out of bounds
    pad_left = max(0, -start)
    pad_right = max(0, end - total_samples)

    # Clamp to valid range
    valid_start = max(0, start)
    valid_end = min(total_samples, end)

    segment = waveform[valid_start:valid_end]

    if pad_left > 0 or pad_right > 0:
        segment = np.pad(
            segment, (pad_left, pad_right),
            mode='constant', constant_values=0.0,
        )

    # Guarantee exact length (handles floating arithmetic edge cases)
    if len(segment) < WINDOW_SAMPLES:
        segment = np.pad(
            segment, (0, WINDOW_SAMPLES - len(segment)),
            mode='constant',
        )
    elif len(segment) > WINDOW_SAMPLES:
        segment = segment[:WINDOW_SAMPLES]
    return segment


def predict_window_tta(sessions, waveform, center_start, n_tta=3):
    """
    Run TTA ensemble inference for a single target window.

    Args:
        sessions: list of ONNX sessions (the fold ensemble)
        waveform: full soundscape waveform (numpy array)
        center_start: start sample index of the target window
        n_tta: number of TTA shifts to use (1, 2, or 3).
               1 = no TTA (center only)
               2 = center + one shift
               3 = all three offsets

    Returns:
        probs: sigmoid probabilities (numpy array of shape (NUM_CLASSES,))
    """
    offsets = TTA_OFFSETS_SAMPLES[:n_tta] if n_tta < 3 else TTA_OFFSETS_SAMPLES
    # For n_tta < 3, prefer the center offset by putting 0 first
    if n_tta == 1:
        offsets = [0]
    elif n_tta == 2:
        offsets = [0, TTA_OFFSETS_SAMPLES[2]]  # center + +2.5s

    all_logits = []
    for offset in offsets:
        segment = extract_shifted_segment(waveform, center_start, offset)
        spec = compute_melspec(segment)
        x = spec[np.newaxis, np.newaxis, :, :].astype(np.float32)
        for sess in sessions:
            logits = sess.run(None, {'input': x})[0][0]
            all_logits.append(logits)

    mean_logits = np.mean(all_logits, axis=0)
    probs = 1.0 / (1.0 + np.exp(-mean_logits))
    return probs

In [6]:
# ═══════════════════════════════════════════════════════════════════
# PROCESS ALL TEST SOUNDSCAPES — with 3× TTA and time-budget fallback
#
# If total projected time exceeds TIME_BUDGET_SECONDS, the loop drops
# to 2× TTA for remaining soundscapes (keeping the best-quality path
# for as many soundscapes as possible). If even 2× is too slow, drops
# to 1× (no TTA). Better a complete submission than a timeout.
# ═══════════════════════════════════════════════════════════════════

TIME_BUDGET_SECONDS = 80 * 60   # hard ceiling — fall back below this
TIME_WARN_SECONDS = 75 * 60      # warn at this point
N_TTA_INITIAL = 3                # start with 3× TTA
N_TTA_FALLBACK_1 = 2             # first fallback: 2× TTA
N_TTA_FALLBACK_2 = 1             # last resort: no TTA

audio_extensions = ('*.ogg', '*.wav', '*.flac', '*.mp3')
audio_files = []
for ext in audio_extensions:
    audio_files.extend(glob.glob(os.path.join(TEST_DIR, ext)))
audio_files = sorted(audio_files)
print(f'Test soundscapes: {len(audio_files)}')

total_start = time.time()
rows = []
timing = {'audio': 0, 'spec_plus_infer': 0, 'windows': 0}
n_tta_current = N_TTA_INITIAL
tta_downgrades = []

for file_idx, audio_path in enumerate(audio_files):
    soundscape_id = os.path.splitext(os.path.basename(audio_path))[0]

    # ── Load audio ─────────────────────────────────────────────────
    t0 = time.perf_counter()
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    timing['audio'] += time.perf_counter() - t0

    total_samples = len(y)
    n_windows = int(np.ceil(total_samples / WINDOW_SAMPLES))

    # ── Predict each window with TTA ───────────────────────────────
    for win_idx in range(n_windows):
        center_start = win_idx * WINDOW_SAMPLES
        end_time_seconds = (win_idx + 1) * int(WINDOW_SECONDS)
        row_id = f'{soundscape_id}_{end_time_seconds}'

        t0 = time.perf_counter()
        probs = predict_window_tta(sessions, y, center_start, n_tta=n_tta_current)
        timing['spec_plus_infer'] += time.perf_counter() - t0
        timing['windows'] += 1

        row = {'row_id': row_id}
        for sp_idx, sp in enumerate(SPECIES_LIST):
            row[sp] = float(probs[sp_idx])
        rows.append(row)

    # ── Time budget check (every soundscape) ───────────────────────
    elapsed = time.time() - total_start
    rate = (file_idx + 1) / elapsed
    remaining = (len(audio_files) - file_idx - 1) / rate if rate > 0 else 0
    est_total = elapsed + remaining

    # Fallback logic: if projected to exceed budget, drop TTA level
    if est_total > TIME_BUDGET_SECONDS and n_tta_current > 1:
        old = n_tta_current
        n_tta_current = max(1, n_tta_current - 1)
        tta_downgrades.append({
            'soundscape_idx': file_idx + 1,
            'elapsed_min': elapsed / 60,
            'est_total_min': est_total / 60,
            'from_tta': old,
            'to_tta': n_tta_current,
        })
        print(f'  ⚠ TIME-BUDGET FALLBACK at soundscape {file_idx+1}/{len(audio_files)}: '
              f'est_total={est_total/60:.1f}min > budget={TIME_BUDGET_SECONDS/60:.0f}min. '
              f'Dropping TTA: {old}× → {n_tta_current}×')

    # Progress reporting
    if (file_idx + 1) % 10 == 0 or file_idx == 0 or file_idx == len(audio_files) - 1:
        warn = '  ⚠' if est_total > TIME_WARN_SECONDS else ' '
        print(f'{warn} [{file_idx+1:>4}/{len(audio_files)}] '
              f'{soundscape_id}: {n_windows} windows | '
              f'TTA={n_tta_current}× | '
              f'Elapsed: {elapsed/60:.1f}min | '
              f'ETA: {remaining/60:.1f}min | '
              f'Total est: {est_total/60:.1f}min')

total_elapsed = time.time() - total_start
n_win = timing['windows']
print(f'\nDone! {len(audio_files)} soundscapes, {n_win} windows in '
      f'{total_elapsed:.1f}s ({total_elapsed/60:.1f}min)')
if n_win > 0:
    print(f'  Per window: audio={timing["audio"]/len(audio_files)*1000:.1f}ms/file, '
          f'spec+infer={timing["spec_plus_infer"]/n_win*1000:.1f}ms/window')

if tta_downgrades:
    print(f'\n⚠ TTA was downgraded {len(tta_downgrades)} time(s) to stay within budget:')
    for d in tta_downgrades:
        print(f'  At soundscape {d["soundscape_idx"]}: '
              f'TTA {d["from_tta"]}× → {d["to_tta"]}× '
              f'(elapsed {d["elapsed_min"]:.1f}min, projected {d["est_total_min"]:.1f}min)')
else:
    print(f'\n✓ Completed with full {N_TTA_INITIAL}× TTA throughout.')

Test soundscapes: 0

Done! 0 soundscapes, 0 windows in 0.0s (0.0min)

✓ Completed with full 3× TTA throughout.


In [7]:
# ═══════════════════════════════════════════════════════════════════
# BUILD SUBMISSION
# ═══════════════════════════════════════════════════════════════════
expected_cols = list(sample_sub.columns)

if rows:
    submission = pd.DataFrame(rows)
    # Ensure exact column match with sample submission
    for col in expected_cols:
        if col not in submission.columns:
            submission[col] = 0.0
    submission = submission[expected_cols]
else:
    # No test files found — use sample_submission as skeleton
    submission = sample_sub.copy()
    print('⚠ No audio files found — using sample submission as placeholder')

# Validate
assert list(submission.columns) == expected_cols, 'Column mismatch!'
print(f'Submission shape: {submission.shape}')
print(f'Expected shape:   ({len(sample_sub)}, {len(expected_cols)})')
print(f'Columns match:    ✓')

# Quick sanity check
pred_vals = submission[SPECIES_LIST].values
print(f'\nPrediction stats:')
print(f'  Mean: {pred_vals.mean():.6f}')
print(f'  Std:  {pred_vals.std():.6f}')
print(f'  Min:  {pred_vals.min():.6f}')
print(f'  Max:  {pred_vals.max():.6f}')

submission.head()

⚠ No audio files found — using sample submission as placeholder
Submission shape: (3, 235)
Expected shape:   (3, 235)
Columns match:    ✓

Prediction stats:
  Mean: 0.004274
  Std:  0.000000
  Min:  0.004274
  Max:  0.004274


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


In [8]:
# ═══════════════════════════════════════════════════════════════════
# SAVE
# ═══════════════════════════════════════════════════════════════════
submission.to_csv('submission.csv', index=False)
print(f'  {submission.shape[0]} rows × {submission.shape[1]} columns')

  3 rows × 235 columns


In [9]:
pd.read_csv('/kaggle/working/submission.csv')

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
